## 0. Setup


In [29]:
import json, socket, socketserver, sqlite3, threading, datetime
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
print("Libraries loaded")

Libraries loaded


In [30]:

MYSQL = {"host": "localhost", "user": "root", "password": "Vyshnav@2005", "port": 3306}
DB_NAME = "epl_project"
MODEL_FOR_GRAPH = "linear_regression"      
ALLOW_SQLITE_FALLBACK = True               

## 1. Create the dataset (synthetic)
320 fictional players × 8 seasons (2018-2025). Players sometimes **transfer between clubs**, and each player's market value reacts to
their stats with a one-year delay plus a personal "hype" factor. That is why actual and predicted values differ from year to year.

In [31]:
SEASONS = list(range(2018, 2026))
CLUBS = ["Arsenal","Aston Villa","Bournemouth","Brentford","Brighton","Chelsea","Crystal Palace","Everton",
         "Fulham","Liverpool","Man City","Man United","Newcastle","Nottm Forest","Tottenham","West Ham",
         "Wolves","Leicester","Southampton","Ipswich"]
STRENGTH = dict(zip(CLUBS, [1.5,1.2,.8,.8,.9,1.3,.8,.7,.8,1.6,1.8,1.2,1.2,.8,1.2,.9,.7,.6,.5,.5]))
POS_P   = {"GK":.10, "DEF":.35, "MID":.35, "FWD":.20}
G_RATE  = {"GK":0.0, "DEF":.03, "MID":.08, "FWD":.30}    
A_RATE  = {"GK":0.0, "DEF":.04, "MID":.12, "FWD":.14}    
POS_MUL = {"GK":.7,  "DEF":.9,  "MID":1.0, "FWD":1.15}
FIRST = ["Marcus","Liam","Noah","Ethan","Lucas","Mason","Oliver","Jack","Leo","Adam","Daniel","Samuel","Ryan",
         "Kai","Felix","Hugo","Theo","Isaac","Jonas","Nico","Mateo","Rafael","Omar","Yusuf","Kofi","Andre",
         "Tomas","Viktor","Eli","Dario"]
LAST = ["Webb","Hartley","Osei","Moreno","Kowal","Brandt","Silva","Novak","Reyes","Lindqvist","Okafor","Duarte",
        "Fenwick","Marsh","Costa","Bauer","Ivanov","Kane-Lowe","Torres","Abara","Petrov","Haynes","Rossi","Mbeki",
        "Larsen","Quinn","Vargas","Dunmore","Salah-Ray","Ferreira"]

def make_data(n_players=320, seed=7):
    rng = np.random.default_rng(seed)
    names = set()
    while len(names) < n_players:
        names.add(f"{rng.choice(FIRST)} {rng.choice(LAST)}")
    names = sorted(names); rng.shuffle(names)
    p_dest = np.array([STRENGTH[c] ** 2 for c in CLUBS]); p_dest = p_dest / p_dest.sum()   # big clubs buy more
    rows = []
    for pid in range(1, n_players + 1):
        club = rng.choice(CLUBS, p=p_dest / p_dest.sum())
        pos = rng.choice(list(POS_P), p=list(POS_P.values()))
        age0 = int(rng.integers(17, 32))
        ability = rng.lognormal(0, .25)         
        hype = rng.normal(1.0, .12)              
        base_min = float(np.clip(rng.normal(2000, 700), 600, 3200))
        prev_raw = None
        for k, season in enumerate(SEASONS):
            if k > 0 and rng.random() < 0.15:    
                choices = [c for c in CLUBS if c != club]
                pr = np.array([STRENGTH[c] ** 2 for c in choices]); club = rng.choice(choices, p=pr / pr.sum())
            age = age0 + k
            fade = 1.0 if age <= 32 else max(.4, 1 - .1 * (age - 32))
            minutes = int(np.clip(rng.normal(base_min * fade, 350), 0, 3420))
            s = STRENGTH[club]
            goals = int(rng.poisson(G_RATE[pos] * minutes / 90 * s * ability))
            assists = int(rng.poisson(A_RATE[pos] * minutes / 90 * s * ability))
            age_f = np.exp(-((age - 25) ** 2) / 70)
            raw = (3 + .9 * goals + .7 * assists + .004 * minutes) * (.4 + age_f) * s * POS_MUL[pos]
            blended = raw if prev_raw is None else .6 * raw + .4 * prev_raw     
            value = max(.5, blended * hype * rng.lognormal(0, .10)); prev_raw = raw
            rows.append((pid, names[pid-1], pos, season, club, age, goals, assists, minutes, round(value, 2)))
    return pd.DataFrame(rows, columns=["player_id","name","position","season","club","age","goals","assists","minutes","market_value"])

raw_df = make_data()
print(raw_df.shape, "->", raw_df.player_id.nunique(), "players x", raw_df.season.nunique(), "seasons")
raw_df.head()

(2560, 10) -> 320 players x 8 seasons


,player_id,name,position,season,club,age,goals,assists,minutes,market_value
0,1,Theo Dunmore,MID,2018,Everton,19,1,3,2916,12.66
1,1,Theo Dunmore,MID,2019,Everton,20,0,1,2384,9.85
2,1,Theo Dunmore,MID,2020,Everton,21,2,2,2622,13.33
3,1,Theo Dunmore,MID,2021,Everton,22,3,3,2396,14.74
4,1,Theo Dunmore,MID,2022,Everton,23,2,0,2260,13.52


## 2. Database (MySQL)
Tables: **players**, **player_seasons** (stats + actual value per season) and **prediction_log** (every prediction the server makes).
If MySQL cannot be reached, the notebook falls back to a local SQLite file so you can still run it (the report must then say SQLite was used).

In [32]:
BACKEND = "mysql"
try:
    import mysql.connector
    _c = mysql.connector.connect(**MYSQL); _cur = _c.cursor()
    _cur.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"); _c.commit(); _c.close()
    print(f"Connected to MySQL, database '{DB_NAME}' is ready")
except Exception as e:
    if not ALLOW_SQLITE_FALLBACK:
        raise
    BACKEND = "sqlite"
    print("!! Could not use MySQL:", e)
    print("!! Falling back to SQLite (file epl_fallback.db). Fix the MYSQL settings above to use MySQL.")

def get_conn():
    if BACKEND == "mysql":
        return mysql.connector.connect(database=DB_NAME, **MYSQL)
    return sqlite3.connect("epl_fallback.db")

def _py(v):                               # numpy -> plain Python (MySQL driver needs this)
    return v.item() if hasattr(v, "item") else v

def run(sql, params=(), many=False):
    """Execute INSERT/CREATE/DELETE. SQL is written with %s placeholders (MySQL style)."""
    conn = get_conn(); cur = conn.cursor()
    if BACKEND == "sqlite": sql = sql.replace("%s", "?")
    (cur.executemany if many else cur.execute)(sql, params)
    conn.commit(); conn.close()

def read(sql, params=()):
    """Execute a SELECT and return a DataFrame."""
    conn = get_conn(); cur = conn.cursor()
    if BACKEND == "sqlite": sql = sql.replace("%s", "?")
    cur.execute(sql, tuple(params))
    out = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
    conn.close()
    return out

auto = "AUTO_INCREMENT" if BACKEND == "mysql" else "AUTOINCREMENT"
pk = "INT" if BACKEND == "mysql" else "INTEGER"
run("CREATE TABLE IF NOT EXISTS players(player_id INT PRIMARY KEY, name VARCHAR(60), position VARCHAR(3))")
run("""CREATE TABLE IF NOT EXISTS player_seasons(
        player_id INT, season INT, club VARCHAR(30), age INT, goals INT, assists INT, minutes INT,
        market_value DOUBLE, PRIMARY KEY (player_id, season),
        FOREIGN KEY (player_id) REFERENCES players(player_id))""")
run(f"""CREATE TABLE IF NOT EXISTS prediction_log(
        id {pk} PRIMARY KEY {auto}, logged_at VARCHAR(25), player_id INT, season INT,
        actual_value DOUBLE, predicted_value DOUBLE)""")

# (re)load the data
run("DELETE FROM player_seasons"); run("DELETE FROM players")
players = raw_df[["player_id","name","position"]].drop_duplicates()
run("INSERT INTO players VALUES (%s,%s,%s)", [tuple(map(_py, r)) for r in players.itertuples(index=False)], many=True)
ps = raw_df[["player_id","season","club","age","goals","assists","minutes","market_value"]]
run("INSERT INTO player_seasons VALUES (%s,%s,%s,%s,%s,%s,%s,%s)", [tuple(map(_py, r)) for r in ps.itertuples(index=False)], many=True)
print(f"Stored {len(players)} players and {len(ps)} player-seasons in {BACKEND.upper()}")

Connected to MySQL, database 'epl_project' is ready
Stored 320 players and 2560 player-seasons in MYSQL


In [33]:
# Read the data back with a SQL JOIN - everything below uses this DataFrame
data = read("""SELECT s.player_id, p.name, p.position, s.season, s.club, s.age,
                       s.goals, s.assists, s.minutes, s.market_value
                FROM player_seasons s JOIN players p ON p.player_id = s.player_id""")
print(data.shape)
read("SELECT club, COUNT(*) AS player_seasons, ROUND(AVG(market_value),1) AS avg_value_M "
     "FROM player_seasons WHERE season = 2025 GROUP BY club ORDER BY avg_value_M DESC LIMIT 5")

(2560, 10)


,club,player_seasons,avg_value_M
0,Man City,33,33.3
1,Arsenal,27,23.6
2,Liverpool,39,23.3
3,Chelsea,25,20.0
4,Tottenham,20,19.0


## 3. Data cleaning checks

In [34]:
print("Missing values:", int(data.isna().sum().sum()))
print("Duplicate player-seasons:", int(data.duplicated(["player_id","season"]).sum()))
print("Seasons per player:", sorted(int(n) for n in data.groupby("player_id").size().unique()))
print("Value range (EUR M):", data.market_value.min(), "to", data.market_value.max())

Missing values: 0
Duplicate player-seasons: 0
Seasons per player: [8]
Value range (EUR M): 0.76 to 236.51


## 4. Data analytics

In [35]:
latest_season = int(data.season.max())
num_cols = ["age","goals","assists","minutes","market_value"]
print("Summary statistics (all player-seasons):")
display(data[num_cols].describe().round(2))
print("Average market value (EUR M) by position and by age group:")
data["age_group"] = pd.cut(data.age, [16,21,25,29,33,45], labels=["17-21","22-25","26-29","30-33","34+"])
display(data.pivot_table(index="position", columns="age_group", values="market_value", aggfunc="mean", observed=True).round(1))
print("Correlation of each statistic with market value:")
display(data[num_cols].corr()["market_value"].drop("market_value").round(2).to_frame("correlation"))

Summary statistics (all player-seasons):


,age,goals,assists,minutes,market_value
count,2560.00,2560.00,2560.00,2560.00,2560.00
mean,27.38,2.93,2.43,1908.99,21.92
std,5.04,4.53,2.71,773.11,19.39
min,17.00,0.00,0.00,0.00,0.76
25%,23.00,0.00,0.00,1366.25,10.18
50%,27.00,1.00,2.00,1932.50,16.50
75%,31.00,4.00,4.00,2474.25,27.74
max,38.00,44.00,29.00,3420.00,236.51


Average market value (EUR M) by position and by age group:


age_group,17-21,22-25,26-29,30-33,34+
position,,,,,
DEF,14.0,18.6,18.4,14.0,7.8
FWD,37.6,45.8,42.9,35.8,16.2
GK,11.0,12.9,11.1,6.8,3.5
MID,16.0,23.8,27.7,18.8,9.1


Correlation of each statistic with market value:


,correlation
age,-0.18
goals,0.81
assists,0.72
minutes,0.41


## 5. Machine learning: predicting market value
Inputs: age, goals, assists, minutes, position, club. Three models are compared on **players the model has never seen**
(a baseline that always predicts the average, Linear Regression, Random Forest).
The predicted line in the graph comes from `MODEL_FOR_GRAPH` using **GroupKFold** by player, so a player's predictions always come from a model that was *not* trained on that player.

In [36]:
NUM, CAT = ["age","goals","assists","minutes"], ["position","club"]
FEATURES = NUM + CAT

def make_model(name):
    est = {"baseline": DummyRegressor(strategy="mean"),
           "linear_regression": LinearRegression(),
           "random_forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=-1)}[name]
    pre = ColumnTransformer([("num", StandardScaler(), NUM), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    return Pipeline([("pre", pre), ("model", est)])

X, y, groups = data[FEATURES], data["market_value"], data["player_id"]
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, y, groups))
rows = []
for name in ["baseline", "linear_regression", "random_forest"]:
    pred = make_model(name).fit(X.iloc[tr], y.iloc[tr]).predict(X.iloc[te])
    rows.append({"model": name, "R2": r2_score(y.iloc[te], pred),
                 "MAE (EUR M)": mean_absolute_error(y.iloc[te], pred),
                 "RMSE (EUR M)": mean_squared_error(y.iloc[te], pred) ** 0.5})
metrics = pd.DataFrame(rows).round(3)
print("Accuracy on unseen players:")
display(metrics)

# 5 models, one per fold; each player is predicted by the fold model that never saw them
fold_models, fold_of = [], {}
for k, (a, b) in enumerate(GroupKFold(n_splits=5).split(X, y, groups)):
    fold_models.append(make_model(MODEL_FOR_GRAPH).fit(X.iloc[a], y.iloc[a]))
    for pid in groups.iloc[b].unique(): fold_of[int(pid)] = k
full_model = make_model(MODEL_FOR_GRAPH).fit(X, y)       # used for any player not in the training data
print(f"Graph model: {MODEL_FOR_GRAPH} | fold models trained: {len(fold_models)}")

Accuracy on unseen players:


,model,R2,MAE (EUR M),RMSE (EUR M)
0,baseline,-0.004,11.598,15.629
1,linear_regression,0.751,5.504,7.775
2,random_forest,0.754,5.238,7.731


Graph model: linear_regression | fold models trained: 5


## 6. NetworkX: club-to-club transfer network


In [37]:
s = data.sort_values(["player_id","season"]).copy()
s["from_club"] = s.groupby("player_id").club.shift()
moves = s[s.from_club.notna() & (s.from_club != s.club)]
G = nx.DiGraph()
for (a, b), n in moves.groupby(["from_club","club"]).size().items():
    G.add_edge(a, b, weight=int(n))
print(f"Transfers in 8 seasons: {len(moves)} | clubs (nodes): {G.number_of_nodes()} | routes (edges): {G.number_of_edges()}")

pr = nx.pagerank(G, weight="weight")
net = pd.DataFrame({
    "bought": pd.Series(dict(G.in_degree(weight="weight"))),
    "sold":   pd.Series(dict(G.out_degree(weight="weight"))),
    "value_bought_M": moves.groupby("club").market_value.sum().round(0),
    "pagerank": pd.Series(pr).round(3)}).fillna(0)
net["net_players"] = net.bought - net.sold
net = net[["bought","sold","net_players","value_bought_M","pagerank"]].astype({"bought":int,"sold":int,"net_players":int})
print("Clubs ranked by PageRank (how 'central' a club is in the transfer flow):")
display(net.sort_values("pagerank", ascending=False).head(8))
print("Most common transfer routes:")
display(pd.DataFrame([(a, b, d["weight"]) for a, b, d in G.edges(data=True)],
                     columns=["from","to","players"]).sort_values("players", ascending=False).head(5).reset_index(drop=True))

Transfers in 8 seasons: 328 | clubs (nodes): 20 | routes (edges): 188
Clubs ranked by PageRank (how 'central' a club is in the transfer flow):


,bought,sold,net_players,value_bought_M,pagerank
Liverpool,44,39,5,1131.0,0.120
Man City,34,45,-11,1031.0,0.093
Arsenal,31,33,-2,766.0,0.088
Man United,26,20,6,415.0,0.076
Aston Villa,23,27,-4,549.0,0.070
Chelsea,18,20,-2,320.0,0.057
Tottenham,20,21,-1,386.0,0.057
Crystal Palace,16,13,3,247.0,0.053


Most common transfer routes:


,from,to,players
0,Liverpool,Man City,11
1,Man City,Liverpool,8
2,Aston Villa,Liverpool,7
3,Man City,Arsenal,7
4,Man United,Liverpool,4


## 7. Socket programming: prediction server and client


In [38]:
HOST, PORT = "127.0.0.1", 5051

def handle_request(req):
    pid = int(req["player_id"])
    rows = pd.DataFrame(req["rows"])
    if rows.empty or not set(FEATURES).issubset(rows.columns):
        return {"status": "error", "message": "rows must contain " + ", ".join(FEATURES)}
    model = fold_models[fold_of[pid]] if pid in fold_of else full_model
    pred = np.clip(model.predict(rows[FEATURES]), 0, None).round(2)
    now = datetime.datetime.now().isoformat(timespec="seconds")
    log = [(now, pid, int(r.season), float(r.market_value), float(p)) for r, p in zip(rows.itertuples(), pred)]
    run("INSERT INTO prediction_log(logged_at, player_id, season, actual_value, predicted_value) VALUES (%s,%s,%s,%s,%s)",
        log, many=True)
    return {"status": "ok", "predictions": [{"season": int(r.season), "predicted": float(p)} for r, p in zip(rows.itertuples(), pred)]}

class Handler(socketserver.StreamRequestHandler):
    def handle(self):
        try:
            resp = handle_request(json.loads(self.rfile.readline().decode()))
        except Exception as e:
            resp = {"status": "error", "message": str(e)}
        self.wfile.write((json.dumps(resp) + "\n").encode())

class Server(socketserver.ThreadingTCPServer):
    allow_reuse_address = True
    daemon_threads = True

if "server" in globals():                  # allows re-running this cell
    server.shutdown(); server.server_close()
server = Server((HOST, PORT), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
print(f"Prediction server running on {HOST}:{PORT}")

def request_prediction(payload, timeout=10):
    with socket.create_connection((HOST, PORT), timeout=timeout) as c:
        c.sendall((json.dumps(payload) + "\n").encode())
        return json.loads(c.makefile("r").readline())

Prediction server running on 127.0.0.1:5051


In [39]:
# quick client test with a deliberately bad request (shows error handling)
print(request_prediction({"player_id": 1, "rows": [{"season": 2025}]}))

{'status': 'error', 'message': 'rows must contain age, goals, assists, minutes, position, club'}


## 8. Interactive interface
1. Choose **club**, **position** and **age range** (ages as of the latest season) and click **Find players**.
2. Choose a player from the list of valid players.
3. The player's details, season statistics and the **actual vs predicted** graph appear below.

In [40]:
def value_chart(h):
    x, actual, pred = h.season.values, h.actual.values, h.predicted.values
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(x, actual, color="#2f7ed8", linewidth=2.5, marker="o", label="Actual value")
    ax.plot(x, pred, color="#f39c12", linewidth=2, linestyle="--", marker="s", label="Predicted value")
    ax.fill_between(x, actual, pred, where=actual >= pred, interpolate=True, color="#e74c3c", alpha=.25, label="Overvalued (actual > predicted)")
    ax.fill_between(x, actual, pred, where=actual < pred, interpolate=True, color="#2ecc71", alpha=.25, label="Undervalued (actual < predicted)")
    ax.set_xticks(x); ax.set_xlabel("Season"); ax.set_ylabel("Market value (EUR million)")
    ax.set_title("Actual vs predicted market value by season"); ax.grid(alpha=.3)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4, frameon=False, fontsize=9)
    fig.tight_layout()
    return fig

def player_report(pid):
    """Reads the player from MySQL, asks the socket server for predictions, returns everything needed to display."""
    h = read("SELECT s.season, s.club, p.position, s.age, s.goals, s.assists, s.minutes, s.market_value, p.name "
             "FROM player_seasons s JOIN players p ON p.player_id = s.player_id WHERE s.player_id = %s ORDER BY s.season", (pid,))
    resp = request_prediction({"player_id": int(pid), "rows": json.loads(h.drop(columns="name").to_json(orient="records"))})
    if resp["status"] != "ok":
        raise RuntimeError(resp["message"])
    h["actual"] = h.market_value
    h["predicted"] = [p["predicted"] for p in resp["predictions"]]
    return h

def show_player(pid):
    h = player_report(pid)
    cur = h.iloc[-1]
    stays = h.groupby((h.club != h.club.shift()).cumsum()).agg(club=("club","first"), a=("season","min"), b=("season","max"))
    history = "  >  ".join(f"{r.club} ({r.a}-{r.b})" for r in stays.itertuples())
    rel = ((h.actual - h.predicted) / h.predicted).mean() * 100
    verdict = "OVERVALUED" if rel > 10 else "UNDERVALUED" if rel < -10 else "fairly valued"
    print(f"PLAYER: {cur['name']}")
    print(f"Position: {cur.position} | Age: {cur.age} | Current club: {cur.club}")
    print(f"Club history: {history}")
    print(f"Value {cur.season}: actual EUR {cur.actual:.1f}M vs predicted EUR {cur.predicted:.1f}M")
    print(f"Verdict over 8 seasons: {verdict} (actual is on average {rel:+.0f}% from predicted)\n")
    t = h[["season","club","age","goals","assists","minutes","actual","predicted"]].copy()
    t["gap (actual-pred)"] = (t.actual - t.predicted).round(2)
    t.columns = ["Season","Club","Age","Goals","Assists","Minutes","Actual (EUR M)","Predicted (EUR M)","Gap (actual-pred)"]
    display(t.set_index("Season"))
    fig = value_chart(h); plt.show()
    return h

def find_players(club, position, age_lo, age_hi):
    return read("SELECT p.player_id, p.name, s.age, s.market_value FROM player_seasons s "
                "JOIN players p ON p.player_id = s.player_id "
                "WHERE s.season = %s AND s.club = %s AND p.position = %s AND s.age BETWEEN %s AND %s "
                "ORDER BY s.market_value DESC", (latest_season, club, position, age_lo, age_hi))

In [41]:
cur_ages = data[data.season == latest_season].age
w_club = widgets.Dropdown(options=sorted(CLUBS), description="Club:")
w_pos  = widgets.Dropdown(options=["GK","DEF","MID","FWD"], value="FWD", description="Position:")
w_age  = widgets.IntRangeSlider(value=[int(cur_ages.min()), int(cur_ages.max())], min=int(cur_ages.min()),
                                max=int(cur_ages.max()), description="Age:", continuous_update=False)
w_btn  = widgets.Button(description="Find players", button_style="primary")
w_play = widgets.Dropdown(options=[], description="Player:", disabled=True, layout=widgets.Layout(width="480px"))
w_info = widgets.HTML()
w_out  = widgets.Output()

def on_find(_):
    w_out.clear_output()
    found = find_players(w_club.value, w_pos.value, *w_age.value)
    if found.empty:
        w_play.options = []; w_play.disabled = True
        w_info.value = "<b style='color:#c0392b'>No players match this club, position and age range. Try different filters.</b>"
        return
    w_info.value = f"<b>{len(found)} valid player(s) found.</b> Select one:"
    w_play.disabled = False
    w_play.options = [(f"{r.name}  (age {r.age}, EUR {r.market_value:.1f}M)", int(r.player_id)) for r in found.itertuples()]

def on_player(change):
    if change["new"] is None: return
    with w_out:
        clear_output(wait=True)
        show_player(change["new"])

w_btn.on_click(on_find)
w_play.observe(on_player, names="value")
display(widgets.VBox([w_club, w_pos, w_age, w_btn, w_info, w_play, w_out]))

## 9. Prediction log (stored in the database by the socket server)

In [42]:
read("SELECT * FROM prediction_log ORDER BY id DESC LIMIT 10")

,id,logged_at,player_id,season,actual_value,predicted_value
0,136,2026-10-26T10:26:44,54,2025,13.87,27.57
1,135,2026-10-26T10:26:44,54,2024,14.89,18.55
2,134,2026-10-26T10:26:44,54,2023,15.71,19.25
3,133,2026-10-26T10:26:44,54,2022,16.87,16.40
4,132,2026-10-26T10:26:44,54,2021,16.50,23.33
5,131,2026-10-26T10:26:44,54,2020,19.52,15.14
6,130,2026-10-26T10:26:44,54,2019,22.42,22.37
7,129,2026-10-26T10:26:44,54,2018,22.40,16.76
8,128,2026-10-26T10:26:32,258,2025,43.44,42.19
9,127,2026-10-26T10:26:32,258,2024,32.81,32.20
